In [1]:

##Using cross-sectional z-score by quarter for all features


In [2]:

import pandas as pd
import numpy as np
from typing import Tuple, Dict
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('fundamentals_with_prices.csv')

df.isna().sum()[df.isna().sum() > 0]
df = df.dropna(subset=["close_price"])
df.isna().sum()[df.isna().sum() > 0]

df = df.sort_values(["ticker", "fiscalDateEnding"])
df["future_return_1y"] = ( df.groupby("ticker")["close_price"].shift(-4) / df["close_price"] - 1)


In [2]:

df = df.sort_values(["ticker","fiscalDateEnding"])

df = df.dropna(subset=["future_return_1y"])
df.groupby("ticker")["future_return_1y"].apply(lambda x: x.tail(4))


ticker      
AAPL    1247    0.312585
        1246   -0.048999
        1245    0.097919
        1244    0.087689
ADBE    55     -0.217249
                  ...   
ZM      1053    0.059351
ZS      1151    0.307795
        1150    0.592194
        1149    0.831628
        1148   -0.012735
Name: future_return_1y, Length: 220, dtype: float64

In [3]:

df["fiscalDateEnding"] = pd.to_datetime(df["fiscalDateEnding"])

df["quarter"] = df["fiscalDateEnding"].dt.to_period("Q")


In [4]:

quarters = sorted(df["quarter"].unique())

test_q = quarters[-1]      # latest quarter
val_q = quarters[-2]       # previous quarter


In [5]:

non_feature_cols = ["ticker", "fiscalDateEnding", 'close_price', 'future_return_1y', 'price_date', 'quarter']
target = "future_return_1y"


In [6]:
feature_cols = df.columns.difference([
    "ticker", "fiscalDateEnding", "close_price",
    "future_return_1y", "price_date", "quarter"
])

In [7]:
# cross-sectional z-score by quarter
df[feature_cols] = (
    df.groupby("quarter")[feature_cols]
      .transform(lambda x: (x - x.mean()) / x.std())
)

In [8]:

test_df = df[df["quarter"] == test_q]

val_df = df[df["quarter"] == val_q]

train_df = df[df["quarter"] < val_q]

In [9]:

df.isna().sum()[df.isna().sum() > 0]

print("Train quarters:", train_df["quarter"].unique())
print("Validation quarter:", val_q)
print("Test quarter:", test_q)

train_df.columns


Train quarters: <PeriodArray>
['2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3',
 '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2',
 '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3']
Length: 19, dtype: period[Q-DEC]
Validation quarter: 2024Q4
Test quarter: 2025Q1


Index(['ticker', 'fiscalDateEnding', 'current_ratio', 'quick_ratio',
       'cash_ratio', 'working_capital_ratio', 'gross_profit_margin',
       'operating_profit_margin', 'net_profit_margin', 'ebitda_margin', 'roa',
       'roe', 'debt_to_equity', 'debt_to_assets', 'equity_ratio',
       'interest_coverage', 'asset_turnover', 'receivables_turnover',
       'inventory_turnover', 'days_inventory_outstanding',
       'days_sales_outstanding', 'operating_expense_ratio', 'sga_ratio',
       'rd_intensity', 'quality_of_earnings', 'depreciation_to_ppe',
       'cost_of_revenue_ratio', 'tax_rate', 'ebit_to_revenue',
       'intangible_asset_ratio', 'goodwill_ratio', 'tangible_asset_ratio',
       'long_term_debt_ratio', 'short_term_debt_ratio', 'debt_composition',
       'log_assets', 'share_growth', 'totalAssets_was_missing',
       'totalCurrentAssets_was_missing',
       'cashAndCashEquivalentsAtCarryingValue_was_missing',
       'cashAndShortTermInvestments_was_missing', 'inventory_was_mi

In [10]:

non_feature_cols = ["ticker", "fiscalDateEnding", 'close_price', 'future_return_1y', 'price_date', 'quarter']
target = "future_return_1y"

X_train = train_df.drop(columns=non_feature_cols)
y_train = train_df[target]

X_val = val_df.drop(columns=non_feature_cols)
y_val = val_df[target]

X_test = test_df.drop(columns=non_feature_cols)
y_test = test_df[target]


In [11]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)



RandomForestRegressor(n_estimators=200, random_state=42)

In [12]:
test_df["pred_return"] = model.predict(X_test)

top10 = test_df.sort_values("pred_return", ascending=False).head(10)

print(top10[["ticker","pred_return"]])

     ticker  pred_return
172    AMAT     0.386403
1124   PANW     0.314329
1029   WDAY     0.286641
785    CPRT     0.286178
1268   CSCO     0.258306
1148     ZS     0.242671
1053     ZM     0.214243
148     ADI     0.060724


In [19]:
from scipy.stats import spearmanr

corr, _ = spearmanr(test_df["future_return_1y"], test_df["pred_return"])

print("Spearman Rank Correlation:", corr)

Spearman Rank Correlation: -0.19047619047619052


In [20]:
top10_each_q = ( test_df.sort_values(["fiscalDateEnding","pred_return"], ascending=[True, False]).groupby("fiscalDateEnding").head(20))

top10 = test_df.sort_values("pred_return", ascending=False).head(20)

print(top10[["ticker","pred_return","future_return_1y"]])


     ticker  pred_return  future_return_1y
172    AMAT     0.386403          0.805082
1124   PANW     0.314329         -0.040397
1029   WDAY     0.286641         -0.329810
785    CPRT     0.286178         -0.299499
1268   CSCO     0.258306          0.323811
1148     ZS     0.242671         -0.012735
1053     ZM     0.214243          0.059351
148     ADI     0.060724          0.491372


In [ ]:

top10 = test_df.sort_values("pred_return", ascending=False).head(10)

top10["future_return_1y"].mean()


In [21]:
from lightgbm import LGBMRegressor, early_stopping

model = LGBMRegressor( n_estimators=300,
                        learning_rate=0.03,
                        max_depth=4,
                        num_leaves=20,
                        min_child_samples=50,
                        subsample=0.7,
                        colsample_bytree=0.7,
                        reg_alpha=1.0,
                        reg_lambda=1.0,
                        random_state=42
                    )


model.fit( X_train,
           y_train,
          eval_set=[(X_val, y_val)],
          eval_metric="rmse",
          callbacks=[early_stopping(50)]
        )


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000924 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9188
[LightGBM] [Info] Number of data points in the train set: 1027, number of used features: 57
[LightGBM] [Info] Start training from score 0.173402
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

LGBMRegressor(colsample_bytree=0.7, learning_rate=0.03, max_depth=4,
              min_child_samples=50, n_estimators=300, num_leaves=20,
              random_state=42, reg_alpha=1.0, reg_lambda=1.0, subsample=0.7)

In [22]:
test_df["pred_return"] = model.predict(X_test, num_iteration=model.best_iteration_)

top10 = test_df.sort_values("pred_return", ascending=False).head(20)

print(top10[["ticker","pred_return"]])



import pandas as pd

imp = pd.DataFrame({"feature": X_train.columns,
                     "importance": model.feature_importances_ }).sort_values("importance", ascending=False)

print(imp.head(20))


     ticker  pred_return
1124   PANW     0.448431
1148     ZS     0.442717
1053     ZM     0.362915
172    AMAT     0.281997
785    CPRT     0.261605
1268   CSCO     0.236378
148     ADI     0.151298
1029   WDAY     0.122645
                              feature  importance
30               long_term_debt_ratio         113
28                     goodwill_ratio          99
23                depreciation_to_ppe          93
3               working_capital_ratio          89
33                         log_assets          85
8                                 roa          79
20                          sga_ratio          71
13                  interest_coverage          69
18             days_sales_outstanding          59
1                         quick_ratio          57
15               receivables_turnover          57
14                     asset_turnover          56
40  currentNetReceivables_was_missing          52
12                       equity_ratio          49
2                        

In [23]:
from scipy.stats import spearmanr

corr, _ = spearmanr(test_df["future_return_1y"], test_df["pred_return"])

print("Spearman Rank Correlation:", corr)

Spearman Rank Correlation: 0.023809523809523815
